# Market prices enrichment

Esta notebook obtiene cotizaciones históricas para enriquecer las operaciones con precio de mercado.

In [0]:
from pyspark.sql import functions as F

In [0]:
OPERACIONES_TABLE = "byma.silver.operaciones_clean"
COTIZACIONES_TABLE = "byma.silver.cotizaciones_historicas"

In [0]:
df_ops = spark.table(OPERACIONES_TABLE)

tickers = (
    df_ops
    .select("simbolo_titulo")
    .distinct()
    .orderBy("simbolo_titulo")
)

print(f"Tickers únicos: {tickers.count()}")

display(tickers)

Tickers únicos: 1445


simbolo_titulo
#MAV150560106
A3
AAL
AALC
AALD
AAP
AAPL
AAPLC
AAPLD
ABBV


In [0]:
tickers_sufijo = (
    tickers
    .withColumn(
        "tipo_sufijo",
        F.when(F.col("simbolo_titulo").endswith("D"), F.lit("D"))
         .when(F.col("simbolo_titulo").endswith("C"), F.lit("C"))
         .otherwise(F.lit(None))
    )
    .filter(F.col("tipo_sufijo").isNotNull())
)

display(tickers_sufijo)

simbolo_titulo,tipo_sufijo
AALC,C
AALD,D
AAPLC,C
AAPLD,D
ABBVD,D
ABEVD,D
ABNBD,D
ABTD,D
ACND,D
ACWID,D


In [0]:
top_tickers = (
    df_ops
    .groupBy("simbolo_titulo")
    .count()
    .orderBy(F.desc("count"))
)

display(top_tickers.limit(100))

simbolo_titulo,count
AL30,8859
AL30D,4589
MSFT,2336
SPY,2311
AMZN,2054
MELI,1881
PAMP,1668
NVDA,1493
YPFD,1485
GOOGL,1453


In [0]:
display(
    top_tickers.filter(
        F.col("simbolo_titulo").rlike(".*[DC]$")
    )
)

simbolo_titulo,count
AL30D,4589
YPFD,1485
GLD,1263
SPYD,457
MSFTD,438
AMD,396
MELID,342
INTC,337
AMZND,283
AL30C,264


## Estrategia de mapeo de tickers

Algunos instrumentos del dataset utilizan sufijos (por ejemplo D o C) que no necesariamente existen como ticker en la API de cotizaciones.

Para los casos detectados se utiliza un mapeo explícito hacia el ticker de referencia. Si un ticker no requiere mapeo, se consulta utilizando su símbolo original.

In [0]:
ticker_mapping = {
    "AL30D": "AL30",
    "AL30C": "AL30",
    "MSFTD": "MSFT",
    "AMZND": "AMZN",
    "AAPLD": "AAPL",
    "NVDAD": "NVDA",
    "GGLAD": "GGAL.BA",
    "PAMP": "PAMP.BA",
    "YPFD": "YPFD.BA",
    "TXAR": "TXAR.BA",
    "ALUA": "ALUA.BA",
}

In [0]:
mapping_rows = [(k, v) for k, v in ticker_mapping.items()]

df_mapping = spark.createDataFrame(
    mapping_rows,
    ["simbolo_titulo", "ticker_referencia"]
)

tickers_consulta = (
    tickers
    .join(df_mapping, "simbolo_titulo", "left")
    .withColumn(
        "ticker_consulta",
        F.coalesce(
            F.col("ticker_referencia"),
            F.col("simbolo_titulo")
        )
    )
)

display(tickers_consulta)

simbolo_titulo,ticker_referencia,ticker_consulta
AXIA,null,AXIA
PAMP,PAMP.BA,PAMP.BA
XLUD,null,XLUD
AAPL,null,AAPL
SHOP,null,SHOP
CELU,null,CELU
IBIT,null,IBIT
PFE,null,PFE
AN29,null,AN29
QQQ,null,QQQ


## Rango de fechas a consultar

In [0]:
rango_fechas = (
    df_ops
    .agg(
        F.min("fecha_operacion").alias("fecha_desde"),
        F.max("fecha_operacion").alias("fecha_hasta")
    )
    .collect()[0]
)

fecha_desde = rango_fechas["fecha_desde"]
fecha_hasta = rango_fechas["fecha_hasta"]

print("Desde:", fecha_desde)
print("Hasta:", fecha_hasta)

Desde: 2026-01-02
Hasta: 2026-03-13


## Descarga de cotizaciones históricas

In [0]:
#%pip install yfinance

In [0]:
#dbutils.library.restartPython()

In [0]:
import time
import pandas as pd
import yfinance as yf

In [0]:
def descargar_cotizacion(ticker, fecha_desde, fecha_hasta, max_intentos=3):
    for intento in range(1, max_intentos + 1):
        try:
            data = yf.download(
                ticker,
                start=str(fecha_desde),
                end=str(fecha_hasta),
                progress=False,
                auto_adjust=False
            )

            if data is not None and len(data) > 0:
                data = data.reset_index()
                data["ticker_consulta"] = ticker
                data["fuente_api"] = "yfinance"
                data["fue_imputado"] = False
                data["razon_imputacion"] = None
                return data

        except Exception as e:
            print(f"Error consultando {ticker}, intento {intento}: {e}")

        time.sleep(2 ** intento)

    return pd.DataFrame()

In [0]:
tickers_test = [
    row["ticker_consulta"]
    for row in tickers_consulta.select("ticker_consulta").distinct().limit(50).collect()
]

cotizaciones = []

for ticker in tickers_test:
    print(f"Descargando {ticker}")
    data = descargar_cotizacion(ticker, fecha_desde, fecha_hasta)
    if len(data) > 0:
        cotizaciones.append(data)

print(f"Tickers con datos: {len(cotizaciones)}")

$XLUD: possibly delisted; no timezone found

1 Failed download:
['XLUD']: possibly delisted; no timezone found


Descargando AXIA
Descargando PAMP.BA
Descargando XLUD


$XLUD: possibly delisted; no timezone found

1 Failed download:
['XLUD']: possibly delisted; no timezone found
$XLUD: possibly delisted; no timezone found

1 Failed download:
['XLUD']: possibly delisted; no timezone found
$AN29: possibly delisted; no timezone found

1 Failed download:
['AN29']: possibly delisted; no timezone found


Descargando AAPL
Descargando SHOP
Descargando CELU
Descargando IBIT
Descargando PFE
Descargando AN29


$AN29: possibly delisted; no timezone found

1 Failed download:
['AN29']: possibly delisted; no timezone found
$AN29: possibly delisted; no timezone found

1 Failed download:
['AN29']: possibly delisted; no timezone found


...

Descargando GFGC7930FE


$GFGC7930FE: possibly delisted; no timezone found

1 Failed download:
['GFGC7930FE']: possibly delisted; no timezone found
$GFGC7930FE: possibly delisted; no timezone found

1 Failed download:
['GFGC7930FE']: possibly delisted; no timezone found
$GFGC7930FE: possibly delisted; no timezone found

1 Failed download:
['GFGC7930FE']: possibly delisted; no timezone found


Descargando MMMD


$MMMD: possibly delisted; no timezone found

1 Failed download:
['MMMD']: possibly delisted; no timezone found
$MMMD: possibly delisted; no timezone found

1 Failed download:
['MMMD']: possibly delisted; no timezone found
$MMMD: possibly delisted; no timezone found

1 Failed download:
['MMMD']: possibly delisted; no timezone found


Tickers con datos: 21


## Normalización de cotizaciones


In [0]:
pdf_prices = []

for pdf in cotizaciones:
    pdf_tmp = pdf.copy()

    if isinstance(pdf_tmp.columns, pd.MultiIndex):
        ticker = [
            col[1]
            for col in pdf_tmp.columns
            if col[1] not in (None, "")
        ][0]

        pdf_tmp.columns = [
            col[0] if col[0] != "Date" else "Date"
            for col in pdf_tmp.columns
        ]

        pdf_tmp["ticker_consulta"] = ticker

    pdf_tmp = pdf_tmp.rename(columns={
        "Date": "fecha",
        "Close": "precio_cierre",
        "Open": "precio_apertura"
    })

    pdf_tmp = pdf_tmp[[
        "ticker_consulta",
        "fecha",
        "precio_cierre",
        "precio_apertura",
        "fuente_api",
        "fue_imputado",
        "razon_imputacion"
    ]]

    pdf_prices.append(pdf_tmp)

pdf_prices = pd.concat(pdf_prices, ignore_index=True)

df_prices = spark.createDataFrame(pdf_prices)

display(df_prices.limit(20))
df_prices.printSchema()

ticker_consulta,fecha,precio_cierre,precio_apertura,fuente_api,fue_imputado,razon_imputacion
AXIA,2026-01-02T00:00:00.000Z,9.25,9.270000457763672,yfinance,false,null
AXIA,2026-01-05T00:00:00.000Z,9.369999885559082,9.1899995803833,yfinance,false,null
AXIA,2026-01-06T00:00:00.000Z,9.59000015258789,9.59000015258789,yfinance,false,null
AXIA,2026-01-07T00:00:00.000Z,9.279999732971191,9.550000190734863,yfinance,false,null
AXIA,2026-01-08T00:00:00.000Z,9.539999961853027,9.4399995803833,yfinance,false,null
AXIA,2026-01-09T00:00:00.000Z,9.609999656677246,9.670000076293945,yfinance,false,null
AXIA,2026-01-12T00:00:00.000Z,9.449999809265137,9.4399995803833,yfinance,false,null
AXIA,2026-01-13T00:00:00.000Z,9.199999809265137,9.449999809265137,yfinance,false,null
AXIA,2026-01-14T00:00:00.000Z,9.420000076293945,9.289999961853027,yfinance,false,null
AXIA,2026-01-15T00:00:00.000Z,9.59000015258789,9.529999732971191,yfinance,false,null


root
 |-- ticker_consulta: string (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- precio_cierre: double (nullable = true)
 |-- precio_apertura: double (nullable = true)
 |-- fuente_api: string (nullable = true)
 |-- fue_imputado: boolean (nullable = true)
 |-- razon_imputacion: void (nullable = true)



In [0]:
print(cotizaciones[0].columns)

MultiIndex([(            'Date',     ''),
            (       'Adj Close', 'AXIA'),
            (           'Close', 'AXIA'),
            (            'High', 'AXIA'),
            (             'Low', 'AXIA'),
            (            'Open', 'AXIA'),
            (          'Volume', 'AXIA'),
            ( 'ticker_consulta',     ''),
            (      'fuente_api',     ''),
            (    'fue_imputado',     ''),
            ('razon_imputacion',     '')],
           names=['Price', 'Ticker'])


## Persistencia de cotizaciones históricas

In [0]:


from pyspark.sql.types import StringType, DoubleType, BooleanType


COTIZACIONES_TABLE = "byma.silver.cotizaciones_historicas"

df_prices_final = (
    df_prices
    .select(
        F.col("ticker_consulta").cast(StringType()).alias("simbolo"),
        F.to_date("fecha").alias("fecha"),
        F.col("precio_cierre").cast(DoubleType()).alias("precio_cierre"),
        F.col("precio_apertura").cast(DoubleType()).alias("precio_apertura"),
        F.col("fuente_api").cast(StringType()).alias("fuente_api"),
        F.col("fue_imputado").cast(BooleanType()).alias("fue_imputado"),
        F.coalesce(
            F.col("razon_imputacion").cast(StringType()),
            F.lit("")
        ).alias("razon_imputacion")
    )
)

(
    df_prices_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(COTIZACIONES_TABLE)
)

In [0]:
cotizaciones_df = spark.table(COTIZACIONES_TABLE)

print("Registros:", cotizaciones_df.count())

display(cotizaciones_df.limit(20))

Registros: 1008


simbolo,fecha,precio_cierre,precio_apertura,fuente_api,fue_imputado,razon_imputacion
AXIA,2026-01-02,9.25,9.270000457763672,yfinance,false,
AXIA,2026-01-05,9.369999885559082,9.1899995803833,yfinance,false,
AXIA,2026-01-06,9.59000015258789,9.59000015258789,yfinance,false,
AXIA,2026-01-07,9.279999732971191,9.550000190734863,yfinance,false,
AXIA,2026-01-08,9.539999961853027,9.4399995803833,yfinance,false,
AXIA,2026-01-09,9.609999656677246,9.670000076293945,yfinance,false,
AXIA,2026-01-12,9.449999809265137,9.4399995803833,yfinance,false,
AXIA,2026-01-13,9.199999809265137,9.449999809265137,yfinance,false,
AXIA,2026-01-14,9.420000076293945,9.289999961853027,yfinance,false,
AXIA,2026-01-15,9.59000015258789,9.529999732971191,yfinance,false,
